# 02. Preprocessing & Feature Engineering

## Superstore Sales Data Mining Project

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.features.builder import FeatureBuilder

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load Data

In [3]:
# Load data
loader = DataLoader()
df = loader.generate_sample_data(n_orders=2000)
print(f"Original shape: {df.shape}")

INFO:src.data.loader:Generated 4982 sample records


Original shape: (4982, 21)


## 2. Data Cleaning

In [4]:
# Initialize cleaner
cleaner = DataCleaner(df)

# Handle missing values
df = cleaner.handle_missing_values(numeric_strategy='median', categorical_strategy='mode')

# Handle duplicates
df, dup_removed = cleaner.handle_duplicates(subset=['Order ID', 'Product ID'])
print(f"Duplicates removed: {dup_removed}")

# Handle outliers
df = cleaner.handle_outliers_iqr(columns=['Sales', 'Profit'], threshold=1.5, method='clip')

# Process dates
df = cleaner.process_dates(['Order Date', 'Ship Date'])

print(f"Cleaned shape: {df.shape}")

INFO:src.data.cleaner:Handling missing values...
INFO:src.data.cleaner:Handling duplicates...
INFO:src.data.cleaner:Removed 1 duplicate rows
INFO:src.data.cleaner:Handling outliers with IQR method (threshold=1.5)...
INFO:src.data.cleaner:Outliers handled: {'Sales': np.int64(0), 'Profit': np.int64(0)}
INFO:src.data.cleaner:Processing date columns...


Duplicates removed: 1


AttributeError: Can only use .dt accessor with datetimelike values

## 3. Feature Engineering

In [ ]:
# Initialize feature builder
builder = FeatureBuilder(df)

# Create RFM features
rfm = builder.create_rfm_features()
print("\nRFM Features:")
print(rfm.head())
print("\nSegment distribution:")
print(rfm['Segment'].value_counts())

In [ ]:
# Create basket data for association rules
basket = builder.create_basket_data(min_items=2)
print(f"\nBasket data shape: {basket.shape}")
print(basket.head())

In [ ]:
# Create customer features
customer_features = builder.create_customer_features()
print(f"\nCustomer features shape: {customer_features.shape}")
print(customer_features.head())

## 4. Visualize RFM

In [ ]:
# Plot RFM segments
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# RFM Score distribution
axes[0].hist(rfm['RFM_Score'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('RFM Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('RFM Score Distribution')

# Segment distribution
segment_counts = rfm['Segment'].value_counts()
axes[1].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Customer Segments')

# RFM scatter
scatter = axes[2].scatter(rfm['Recency'], rfm['Monetary'], c=rfm['Frequency'], cmap='viridis', alpha=0.6, s=20)
axes[2].set_xlabel('Recency (days)')
axes[2].set_ylabel('Monetary ($)')
axes[2].set_title('RFM Analysis')
plt.colorbar(scatter, ax=axes[2], label='Frequency')

plt.tight_layout()
plt.savefig('../outputs/figures/06_rfm_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: outputs/figures/06_rfm_analysis.png")

## 5. Save Processed Data

In [ ]:
# Save all features
df.to_csv('../data/processed/02_cleaned_data.csv', index=False)
rfm.to_csv('../data/processed/02_rfm_features.csv', index=False)
basket.to_csv('../data/processed/02_basket_data.csv', index=False)
customer_features.to_csv('../data/processed/02_customer_features.csv', index=False)

print("Preprocessing & Feature Engineering completed!")
print("Saved:")
print("  - data/processed/02_cleaned_data.csv")
print("  - data/processed/02_rfm_features.csv")
print("  - data/processed/02_basket_data.csv")
print("  - data/processed/02_customer_features.csv")